# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library, referencing all dataset entities by their `@id` in line with best practices for FAIR and Croissant-compliant data workflows.

### Dataset Source
The dataset source is described in Croissant JSON-LD format at the following URL: https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and records into memory using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata # Access as object, not as dictionary

# Print general description
print(f"{metadata.name}: {metadata.description}")

### Dataset Curation Details
* Citation: Mugotitsa, B, Maina, D, Owoko, H, Akinyi, L, Greenfield, J, Todd, J, Kavu, T and Kiragga, A 2026 A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa. Frontiers.
* License: https://opendatacommons.org/licenses/by/1-0/
* Data Collection: Primary survey responses from Kilifi County, focusing on mental health indicators (GAD-7, PHQ-9, PSQ) and demographics.
* Data Release Date: 2026-07-17

## 2. Data Overview
List available record sets and their field (column) `@id` values.

**Note**: All references use Croissant `@id`s. To ensure reproducibility and schema-awareness, always use these for all data access, not field names.

In [ ]:
# Enumerate all record sets and their fields by @id
record_sets = list(dataset.record_sets)

print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []
fields_by_record_set_id = dict()

for rs in record_sets:
    print(f"Record set @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields (by @id):")
    field_ids = []
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        print(f"    - {field_id}")
        field_ids.append(field_id)
    fields_by_record_set_id[rs['@id']] = field_ids
    print()

## 3. Data Extraction
Load the data from each record set into a Pandas DataFrame for analysis, referencing each entity by their `@id`.

Replace the below variable assignments as appropriate based on the output of the previous step.

In [ ]:
# List of record set @ids obtained above
record_sets = record_set_ids.copy()
dataframes = {}

for record_set_id in record_sets:
    # Load all records from the record set by @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record_set @id: {record_set_id}\n")

# Show one of the DataFrames (choose the main survey responses record set if available)
if record_sets:
    main_record_set_id = record_sets[0]  # Pick first as example, adjust if needed
    print(f"Available columns (@id) in {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's apply standard EDA to your dataset, using both numeric and categorical fields. For demonstration, we choose a record set and its numeric and group (categorical) fields below. Update these variables as necessary to fit your schema.

In [ ]:
# Customize these IDs to match your schema (see data overview output)
record_set_id = main_record_set_id  # e.g., 'cr:MainRecordSet'

# Find plausible numeric and group (categorical) field @ids from the chosen record set
fields = fields_by_record_set_id[record_set_id]

# Example guesswork for likely numeric and group field IDs
plausible_numeric_fields = [f for f in fields if 'score' in f or 'age' in f or 'gad7' in f or 'phq9' in f or 'psq' in f]
plausible_group_fields = [f for f in fields if 'gender' in f or 'village' in f or 'education' in f or 'marital' in f]

# Set the field IDs for analysis (update as per your actual IDs):
if plausible_numeric_fields:
    numeric_field = plausible_numeric_fields[0]
else:
    numeric_field = fields[0]
if plausible_group_fields:
    group_field = plausible_group_fields[0]
else:
    group_field = fields[1] if len(fields) > 1 else fields[0]

print(f"Selected numeric field @id: {numeric_field}")
print(f"Selected group field @id: {group_field}")

df = dataframes[record_set_id].copy()

# Ensure numeric field
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
# Drop missing values for clean EDA
eda_df = df.dropna(subset=[numeric_field])

# Filtering
threshold = eda_df[numeric_field].mean()  # Use mean as dynamic threshold
filtered_df = eda_df[eda_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df[[numeric_field, group_field]].head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized", group_field]].head())

# Grouping by group field
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and compare means by group field if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(eda_df[numeric_field], bins=15, kde=True)
plt.title(f"Distribution of {numeric_field} (@id)")
plt.xlabel(numeric_field)
plt.show()

if group_field in eda_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field, data=eda_df)
    plt.title(f"{numeric_field} Distribution by {group_field} (@id)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We have demonstrated how to load, enumerate, and analyze a FAIR² dataset using Croissant's rich metadata structure, referencing all record sets and fields by their unique `@id`. This workflow ensures robust, semantics-aware, and schema-driven data science with `mlcroissant` while remaining extendable to additional EDA, machine learning, or integration tasks for reproducible and FAIR research.

Key takeaways:
* Use Croissant `@id` exclusively when handling data entities. This maintains integrity with the underlying schema and supports downstream semantic analysis.
* The dataset contains demographic and mental health survey data, allowing for grouped analyses (e.g., average GAD-7 by gender or village).
* The mlcroissant API supports direct, schema-guided access to records, metadata, and field IDs for trustworthy data science workflows.

**Next steps**: Explore other columns, join with other record sets by common keys if needed, and develop further analyses or predictive models (always referencing data by their Croissant `@id`).